In [1]:
!pip install shap transformers datasets scikit-learn

In [2]:
import os
import json
import re
import torch
import shap
import numpy as np
import pandas as pd
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import logging

# Hide harmless HuggingFace warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR) 

# 1. Setup paths
DATA_PATH = '/kaggle/input/notebooks/srivarshitha16/data-preparation-inlp-project' 
MODEL_PATH = '/kaggle/input/notebooks/srivarshitha16/experimenting-with-fusion-architectures'
GRAPH_PATH = '/kaggle/input/notebooks/srivarshitha16/gnn-pre-training'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Load data and skill vocabulary
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_clean.csv'))

with open(os.path.join(DATA_PATH, 'skill_vocab.json'), 'r') as f:
    skill_vocab = json.load(f)
skill_to_id = {skill: i for i, skill in enumerate(skill_vocab)}

def extract_skill_ids(text, skill_dict, skill_map):
    if not isinstance(text, str): return []
    text_lower = text.lower()
    return [skill_map[skill] for skill in skill_dict if re.search(r'\b' + re.escape(skill) + r'\b', text_lower)]

# 3. Define architecture
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MoE_GraphCrossEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings, freeze_transformer=False):
        super(MoE_GraphCrossEncoder, self).__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        
        t_dim = self.transformer.config.hidden_size
        g_dim = pretrained_embeddings.shape[1] 
        
        self.text_expert = nn.Sequential(nn.Linear(t_dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
        self.graph_expert = nn.Sequential(nn.Linear(g_dim * 2, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
        self.gating_network = nn.Sequential(nn.Linear(t_dim + (g_dim * 2), 128), nn.ReLU(), nn.Linear(128, 2), nn.Softmax(dim=1))

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        cls_embedding = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :] 
        batch_size = cls_embedding.size(0)
        res_graph_embs, jd_graph_embs = [], []
        
        for i in range(batch_size):
            r_embs = self.skill_embeddings(torch.tensor(res_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if res_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            j_embs = self.skill_embeddings(torch.tensor(jd_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if jd_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            res_graph_embs.append(r_embs); jd_graph_embs.append(j_embs)
            
        r_tensor, j_tensor = torch.stack(res_graph_embs), torch.stack(jd_graph_embs)
        text_score = self.text_expert(cls_embedding)
        graph_score = self.graph_expert(torch.cat([r_tensor, j_tensor], dim=1))
        gate_weights = self.gating_network(torch.cat([cls_embedding, r_tensor, j_tensor], dim=1))
        
        final_score = (gate_weights[:, 0].unsqueeze(1) * text_score) + (gate_weights[:, 1].unsqueeze(1) * graph_score)
        return final_score.squeeze(-1), gate_weights, text_score.squeeze(-1), graph_score.squeeze(-1)

print("MoE Architecture successfully defined.")

# 4. Load model weights
try:
    # 1. Load the Graph Embeddings explicitly from Notebook 2's folder
    emb_path = os.path.join(GRAPH_PATH, 'skill_embeddings.npy')
    pretrained_skill_embs = torch.tensor(np.load(emb_path), dtype=torch.float32)
    
    # 2. Initialize Model
    moe_model = MoE_GraphCrossEncoder(MODEL_NAME, pretrained_skill_embs).to(device)
    
    # 3. Load the RoBERTa weights from Notebook 3
    weights_path = os.path.join(MODEL_PATH, 'hyper_optimized_moe.pth')
    moe_model.load_state_dict(torch.load(weights_path, map_location=device))
    moe_model.eval()
    print("Model Weights Loaded Successfully.")
except Exception as e:
    print(f"Error loading weights: {e}")

# 5. Dynamic SHAP wrapper
CURRENT_JD_TEXT = "" 
CURRENT_JD_SKILLS = []

def shap_predict_wrapper(resume_texts_array):
    """
    SHAP passes masked versions of a resume here.
    We extract skills, tokenize, and return ONLY the final score.
    """
    scores = []
    batch_size = 16 
    
    for i in range(0, len(resume_texts_array), batch_size):
        batch_texts = resume_texts_array[i:i+batch_size]
        
        batch_res_skills = [extract_skill_ids(str(text), skill_vocab, skill_to_id) for text in batch_texts]
        batch_jd_skills = [CURRENT_JD_SKILLS for _ in range(len(batch_texts))] 
        
        encodings = tokenizer(
            list(batch_texts), 
            [CURRENT_JD_TEXT] * len(batch_texts), 
            max_length=512, padding='max_length', truncation=True, return_tensors='pt'
        )
        
        with torch.no_grad():
            final_score, _, _, _ = moe_model(
                encodings['input_ids'].to(device), 
                encodings['attention_mask'].to(device), 
                batch_res_skills, 
                batch_jd_skills
            )
            scores.extend(final_score.cpu().numpy())
            
    return np.array(scores)

print("SHAP Dynamic Wrapper defined.")

Using device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

MoE Architecture successfully defined.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

The following layers were not sharded: encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.key.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.value.weight, embeddings.LayerNorm.weight, pooler.dense.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.key.weight, encoder.layer.*.output.dense.bias, embeddings.position_embeddings.weight, pooler.dense.bias, encoder.layer.*.intermediate.dense.weight, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.query.bias, embeddings.word_embeddings.weight, embeddings.LayerNorm.bias


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Model Weights Loaded Successfully.
SHAP Dynamic Wrapper defined.


In [3]:
# 1. Select a single candidate to test
sample_idx = 0
candidate_resume = str(test_df['resume_text'].iloc[sample_idx])

# 2. Set the global JD context
CURRENT_JD_TEXT = str(test_df['job_description_text'].iloc[sample_idx])
CURRENT_JD_SKILLS = extract_skill_ids(CURRENT_JD_TEXT, skill_vocab, skill_to_id)

print(f"Testing SHAP on Resume {sample_idx}...")
print(f"True Label: {test_df['score'].iloc[sample_idx]}")

# 3. Initialize the Explainer
# SHAP will use the RoBERTa tokenizer to figure out how to mask the words
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(shap_predict_wrapper, masker)

# 4. Generate SHAP Values 
shap_values = explainer([candidate_resume])

# 5. Visualize
shap.plots.text(shap_values)

Token indices sequence length is longer than the specified maximum sequence length for this model (1620 > 512). Running this sequence through the model will result in indexing errors


Testing SHAP on Resume 0...
True Label: 0.0


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [01:07, 67.25s/it]               


In [4]:
import pickle
import warnings
from torch.nn.utils.rnn import pad_sequence
class GraphEnhancedCrossEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        self.mlp = nn.Sequential(
            nn.Linear(self.transformer.config.hidden_size + (pretrained_embeddings.shape[1] * 2), 256),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        cls_embedding = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        res_graph_embs, jd_graph_embs = [], []
        for i in range(cls_embedding.size(0)):
            r_embs = self.skill_embeddings(torch.tensor(res_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if res_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            j_embs = self.skill_embeddings(torch.tensor(jd_skill_ids[i], dtype=torch.long, device=cls_embedding.device)).mean(dim=0) if jd_skill_ids[i] else torch.zeros(self.skill_embeddings.embedding_dim, device=cls_embedding.device)
            res_graph_embs.append(r_embs); jd_graph_embs.append(j_embs)
        return self.mlp(torch.cat([cls_embedding, torch.stack(res_graph_embs), torch.stack(jd_graph_embs)], dim=1)).squeeze(-1)

class CrossAttention_GraphEncoder(nn.Module):
    def __init__(self, transformer_name, pretrained_embeddings):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.skill_embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        self.d_text, self.d_graph = self.transformer.config.hidden_size, pretrained_embeddings.shape[1]
        
        self.graph_projector = nn.Linear(self.d_graph, self.d_text)
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.d_text, num_heads=4, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(self.d_text * 2, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, input_ids, attention_mask, res_skill_ids, jd_skill_ids):
        text_sequence = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state 
        cls_token = text_sequence[:, 0, :] 
        
        combined_graph_seqs = []
        for i in range(text_sequence.size(0)):
            all_ids = res_skill_ids[i] + jd_skill_ids[i]
            embs = self.skill_embeddings(torch.tensor(all_ids, dtype=torch.long, device=text_sequence.device)) if all_ids else torch.zeros((1, self.d_graph), device=text_sequence.device)
            combined_graph_seqs.append(embs)
            
        padded_graph = pad_sequence(combined_graph_seqs, batch_first=True) 
        projected_graph = self.graph_projector(padded_graph)
        
        attn_output, _ = self.cross_attention(query=text_sequence, key=projected_graph, value=projected_graph)
        return self.mlp(torch.cat([cls_token, attn_output[:, 0, :]], dim=1)).squeeze(-1)


#  LOAD ALL TOKENIZERS AND MODELS
print("Loading MiniLM Tokenizer and Baseline/Cross-Attention Models...")
tokenizer_minilm = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer_roberta = AutoTokenizer.from_pretrained('roberta-base')

# Baseline (GraphEnhancedCrossEncoder)
baseline_model = GraphEnhancedCrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', pretrained_skill_embs).to(device)
baseline_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'best_fusion_model.pth'), map_location=device))
baseline_model.eval()

# Cross-Attention
ca_model = CrossAttention_GraphEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', pretrained_skill_embs).to(device)
ca_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'best_crossattn_model.pth'), map_location=device))
ca_model.eval()


# 3. DYNAMIC SHAP WRAPPER FACTORY
def create_shap_wrapper(model, tokenizer, is_moe=False):
    def wrapper(resume_texts_array):
        scores = []
        batch_size = 16 
        for i in range(0, len(resume_texts_array), batch_size):
            batch_texts = resume_texts_array[i:i+batch_size]
            batch_res_skills = [extract_skill_ids(str(text), skill_vocab, skill_to_id) for text in batch_texts]
            batch_jd_skills = [CURRENT_JD_SKILLS for _ in range(len(batch_texts))] 
            
            encodings = tokenizer(
                list(batch_texts), [CURRENT_JD_TEXT] * len(batch_texts), 
                max_length=512, padding='max_length', truncation=True, return_tensors='pt'
            )
            
            with torch.no_grad():
                if is_moe:
                    final_score, _, _, _ = model(encodings['input_ids'].to(device), encodings['attention_mask'].to(device), batch_res_skills, batch_jd_skills)
                else:
                    final_score = model(encodings['input_ids'].to(device), encodings['attention_mask'].to(device), batch_res_skills, batch_jd_skills)
                scores.extend(final_score.cpu().numpy())
        return np.array(scores)
    return wrapper


# 4. SELECT 15 CASE STUDIES
np.random.seed(42)
sample_indices = test_df[test_df['resume_text'].str.split().str.len() > 100].sample(15).index.tolist()

all_shap_results = {
    'Baseline': [],
    'MoE': [],
    'CrossAttention': []
}

print(f"Starting SHAP analysis on {len(sample_indices)} candidates across 3 models.")
print("This will require heavy computation. Please wait...")

# 5. THE MASTER LOOP
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    for idx in sample_indices:
        print(f"\nProcessing Candidate Index: {idx}")
        
        # Truncate text
        raw_res = str(test_df['resume_text'].loc[idx])
        candidate_resume = " ".join(raw_res.split()[:350]) 
        
        raw_jd = str(test_df['job_description_text'].loc[idx])
        CURRENT_JD_TEXT = " ".join(raw_jd.split()[:100])
        CURRENT_JD_SKILLS = extract_skill_ids(CURRENT_JD_TEXT, skill_vocab, skill_to_id)
        
        # --- Evaluate Baseline ---
        print("  -> Running Baseline (GraphEnhancedCrossEncoder)...")
        explainer_base = shap.Explainer(create_shap_wrapper(baseline_model, tokenizer_minilm, False), shap.maskers.Text(tokenizer_minilm))
        shap_base = explainer_base([candidate_resume])
        all_shap_results['Baseline'].append({'idx': idx, 'shap_values': shap_base})
        
        # --- Evaluate Cross-Attention ---
        print("  -> Running Cross-Attention...")
        explainer_ca = shap.Explainer(create_shap_wrapper(ca_model, tokenizer_minilm, False), shap.maskers.Text(tokenizer_minilm))
        shap_ca = explainer_ca([candidate_resume])
        all_shap_results['CrossAttention'].append({'idx': idx, 'shap_values': shap_ca})
        
        # --- Evaluate MoE ---
        print("  -> Running MoE...")
        explainer_moe = shap.Explainer(create_shap_wrapper(moe_model, tokenizer_roberta, True), shap.maskers.Text(tokenizer_roberta))
        shap_moe = explainer_moe([candidate_resume])
        all_shap_results['MoE'].append({'idx': idx, 'shap_values': shap_moe})

# 6. SAVE TO DISK
with open('/kaggle/working/all_shap_results.pkl', 'wb') as f:
    pickle.dump(all_shap_results, f)

print("\nSHAP analysis complete and saved to 'all_shap_results.pkl'.")

Loading MiniLM Tokenizer and Baseline/Cross-Attention Models...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

PartitionExplainer explainer: 2it [00:31, 31.06s/it]               


  -> Running MoE...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:43, 43.24s/it]               


SHAP analysis complete and saved to 'all_shap_results.pkl'.
